In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )

Rdair=Co.Rdair()


In [ ]:
%%time
nsteps=None
case, process_ncdata  = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False
#case, process_ncdata  = 'cam77_dyamond1_prod1'    , False
#case , process_ncdata = 'xy-rdg-mm-front'    , True
A = futi.read_case( case=case, nsteps=nsteps ) # , nsteps = 31*8 )

zlev, lat, lon = A.zlev, A.lat, A.lon



In [ ]:
lonW,lonE=0,60
latS,latN=-65,-40
zlev0 = 23_000.

z0=np.argmin(np.abs( zlev - zlev0 ) )
y0,y1=np.argmin(np.abs( lat - latS ) ),np.argmin(np.abs( lat - latN ) )
x0,x1=np.argmin(np.abs( lon - lonW ) ),np.argmin(np.abs( lon - lonE ) )

importlib.reload(auti)
print(y0,y1,z0)
epwp_v = A.rho_epwp[:,z0,y0:y1+1,x0:x1+1].flatten()

In [ ]:
x = epwp_v  #np.abs(upwp)                      # magnitudes
x = x[np.isfinite(x)]                 # remove NaN/inf
x = x[x > 0.0]                        # log scale cannot use zero

# log-spaced bins
nbins = 50
bins = np.logspace(np.log10(x.min()), np.log10(x.max()), nbins + 1)
edges = bins

# bin centers: geometric mean is best for log bins
centers = np.sqrt(edges[:-1] * edges[1:])


In [ ]:
# Which bin each value falls into
ibin = np.digitize(x, bins) - 1

# Sum of |upwp| within each bin
bin_sum = np.zeros(nbins)
bin_sum_I = np.zeros(nbins)

Indc = 0.*x + 1.0
del_x = np.diff(edges)

for i in range(nbins):
    m = ibin == i
    bin_sum[i] = np.sum(x[m])
for i in range(nbins):
    m = ibin == i
    bin_sum_I[i] = np.sum(Indc[m])

# Fractional contribution of each bin to total |upwp|
frac = bin_sum / np.sum(bin_sum)
pdf=bin_sum_I / del_x # np.sum(bin_sum_I ) #/del_x / (len(x))
print( "np.sum(pdf) ",np.sum(pdf) )
pdf=pdf/np.sum(pdf)
print( "np.sum(pdf) ",np.sum(pdf) )

# sort ascending
xs = np.sort(x)

# cumulative sum from the top tail downward
tail_sum = np.cumsum(xs[::-1])[::-1]
tail_frac = tail_sum / tail_sum[0]


In [ ]:
py,px,size=1,3,6.

fig, axs = plt.subplots(py,px,figsize=np.asarray([px*size,py*0.8*size]) )

ax=axs[0]
ax.plot(centers, pdf , marker='o', linestyle='-')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylabel('Probability Density')
ax.set_xlabel('Abs. Mom. Flux. (Pa)')
ax.set_title('PDF of Abs. Mom. Flux')


ax=axs[1]
m = frac > 0
ax.plot(centers[m], frac[m], marker='o')

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlabel('Abs. mom. flux (Pa)')
ax.set_ylabel('Fraction of total AMF')
ax.set_title('Contribution of each bin to total Abs. Mom. Flux')


ax=axs[2]
ax.plot(xs, tail_frac)

ax.set_xscale('log')
ax.set_xlabel('X (Pa)')
ax.set_ylabel(r'Fraction of total AMF from values >= X')
ax.set_title(r'Tail contribution to total AMF')



In [ ]:
ioo=600_000
for poo in [.95,.9,.8,.6,.5,.3]:
    ioo=np.argmin( np.abs(tail_frac-poo) )
    print(xs[ioo], tail_frac[ioo], (len(xs)-ioo)/len(xs), len(xs)-ioo )



In [ ]:
zeta_x = A.zeta[:,:,y0:y1,x0:x1]

In [ ]:
zeta_zy = np.mean( np.mean(zeta_x, axis=0), axis=2)
zeta_prof = np.mean( np.mean( np.mean(zeta_x, axis=0), axis=2), axis=1 )
zeta_zy.shape

In [ ]:
plt.contourf( lat[y0:y1],zlev,zeta_zy)
plt.colorbar()